# Capacity control: a plain ABMIL as large as the RoPE model

**The question:** is the RoPE model's gain over our ABMIL (notebook 15) a gain
from self-attention and tile positions, or just from having four times the
parameters?

Notebook 15's two transformer arms have 4.2 M parameters against the baseline
ABMIL's 1.05 M. This notebook trains the baseline's `GatedABMIL` widened to the
same size: its hidden layer goes from 512 to 2046 units (attention stays 256),
which gives 4,205,305 parameters against the RoPE model's 4,205,319. This is the
control the RoFormer-MIL paper used (its "ABMIL 4.2M"). Nothing else changes:
the packed tissue set, the training recipe, the protocols and seeds, and the
decision rule are notebook 12's and 15's.

Comparisons, each on the same slides with a 95% paired bootstrap interval:

| Comparison | Asks |
|---|---|
| wide ABMIL - ABMIL | does capacity alone help? |
| RoPE model - wide ABMIL | does self-attention with positions help at equal size? |
| transformer - wide ABMIL | does self-attention without positions help at equal size? |

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

functools, os, time   building the arm's model, paths, timing.
pandas                tables.
tqdm                  progress over folds and seeds.
src.features          the packed set, and where predictions are saved.
src.mil               the models, protocols, saved predictions and comparisons.
src.paths             where folds.csv is, locally and on Kaggle.
"""

import functools
import os
import time

import pandas as pd
from tqdm.auto import tqdm

from src.features import (
    open_packed,  # the packed set: features memory-mapped, index, starts, ends, tiles table
    predictions_file,  # where a model's predictions on a set are saved
)
from src.mil import (
    CROSS_SITE,  # the two cross-site directions: name -> (training hospital, test hospital)
    GatedABMIL,  # the baseline's model; hidden sets its width
    TileTransformerABMIL,  # notebook 15's model, built here only to count its parameters
    compare_runs,  # two saved prediction tables of one protocol: QWKs, difference, paired bootstrap interval
    cross_site,  # every slide of one hospital to every slide of the other, once per seed
    cross_validate,  # the mixed protocol on the Stage 4 folds
    grade_distribution,  # share of slides per grade, true against predicted
    load_bags,  # every slide's features into RAM
    prediction_columns,  # the prediction columns of a saved table
    save_predictions,  # predictions to <version>/predictions/<name>.parquet
    seed_summary,  # QWK per seed and their mean and spread
)
from src.paths import (
    DATA_DIR,  # folds.csv lives here locally
    find_input,  # finds folds.csv
)

In [3]:
"""Settings, labels, and the parameter counts that fix the wide ABMIL's size.

1. WIDE_HIDDEN is the hidden width that matches the RoPE model's parameter count
   (attention width stays 256). The counts are printed, not assumed.
2. The arm is saved as "abmil_wide"; the other three models' predictions come
   from notebooks 12 and 15, which must have been run.
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
WIDE_HIDDEN = 2046
ARM = "abmil_wide"
OTHERS = ["abmil", "transformer", "roformer"]
PROTOCOLS = ["mixed", *CROSS_SITE]


def count(model):
    return sum(p.numel() for p in model.parameters())


print(f"parameters: ABMIL {count(GatedABMIL()):,}; wide ABMIL {count(GatedABMIL(hidden=WIDE_HIDDEN)):,}; "
      f"RoPE model {count(TileTransformerABMIL(rope=True)):,}")

packed = open_packed(VERSION)
folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
labels = folds.loc[packed.index["slide_id"]]
missing = [f"{n}-{p}" for n in OTHERS for p in PROTOCOLS if not os.path.exists(predictions_file(VERSION, f"{n}-{p}"))]
if missing:
    raise FileNotFoundError(f"predictions missing: {missing}; run notebooks 12 and 15 first")
print(f"{VERSION}: {len(labels):,} slides")

parameters: ABMIL 1,052,935; wide ABMIL 4,205,305; RoPE model 4,205,319
folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
uni2h-L1-224-t144-hsv-v1: 10,615 slides


## Train and predict

The wide ABMIL under the three protocols, exactly as the baseline: five folds
for mixed, five seeds for each cross-site direction.

In [4]:
"""Train and predict the wide ABMIL under every protocol; save the predictions."""

bags = load_bags(packed)
model_fn = functools.partial(GatedABMIL, hidden=WIDE_HIDDEN)
started = time.perf_counter()
predicted = {"mixed": cross_validate(bags, labels, "mixed", progress=tqdm, model_fn=model_fn)}
for name, (train_hospital, test_hospital) in CROSS_SITE.items():
    predicted[name] = cross_site(bags, labels, train_hospital, test_hospital, progress=tqdm, model_fn=model_fn)
for protocol, values in predicted.items():
    save_predictions(VERSION, f"{ARM}-{protocol}", labels, values)
print(f"{ARM}: trained in {(time.perf_counter() - started) / 60:.1f} min")

abmil_wide: trained in 53.9 min


## Scores

Every model's QWK per protocol (seed mean and range for cross-site), then the
three comparisons with paired bootstrap intervals.

In [5]:
"""Score all four models, then compare the wide ABMIL with the baseline and with both transformer arms."""


def table(name, protocol):
    return pd.read_parquet(predictions_file(VERSION, f"{name}-{protocol}"))


scores = []
for name in ["abmil", ARM, "transformer", "roformer"]:
    row = {"model": name}
    for protocol in PROTOCOLS:
        t = table(name, protocol)
        row[protocol] = seed_summary(t["isup_grade"], t[prediction_columns(t)])["mean"]
    scores.append(row)
print(pd.DataFrame(scores).round(4).to_string(index=False))

pairs = [("abmil", ARM), (ARM, "roformer"), (ARM, "transformer")]
rows = []
for first, second in pairs:
    for protocol in PROTOCOLS:
        for row in compare_runs(table(first, protocol), table(second, protocol), protocol):
            rows.append({"comparison": f"{second} - {first}", **row})
comparison = pd.DataFrame(rows)
print()
print(comparison.round(4).to_string(index=False))

      model  mixed  radboud_to_karolinska  karolinska_to_radboud
      abmil 0.8643                 0.6316                 0.7800
 abmil_wide 0.8639                 0.6422                 0.7714
transformer 0.8714                 0.6617                 0.7508
   roformer 0.8740                 0.7189                 0.7660

              comparison              protocol     slides     n  first  second  second seed std  difference  95% low  95% high
      abmil_wide - abmil                 mixed        all 10615 0.8643  0.8639           0.0000     -0.0005  -0.0067    0.0057
      abmil_wide - abmil                 mixed    radboud  5160 0.8240  0.8301           0.0000      0.0062  -0.0041    0.0170
      abmil_wide - abmil                 mixed karolinska  5455 0.8807  0.8713           0.0000     -0.0094  -0.0173   -0.0016
      abmil_wide - abmil radboud_to_karolinska        all  5455 0.6316  0.6422           0.0315      0.0105   0.0008    0.0212
      abmil_wide - abmil karolinska_to_

## Predicted grades

Share of slides per predicted grade, seeds pooled, for each cross-site
direction.

In [6]:
"""Predicted-grade distribution per cross-site direction: true, then each model."""

for protocol in CROSS_SITE:
    base = table("abmil", protocol)
    shares = {"true": grade_distribution(base["isup_grade"], base[prediction_columns(base)])["true"]}
    for name in ["abmil", ARM, "transformer", "roformer"]:
        run = table(name, protocol).loc[base.index]
        shares[name] = grade_distribution(run["isup_grade"], run[prediction_columns(run)])["predicted"]
    print(f"\n{protocol}: share of slides per grade")
    print(pd.DataFrame(shares).T.round(3).to_string())


radboud_to_karolinska: share of slides per grade
grade            0      1      2      3      4      5
true         0.353  0.333  0.122  0.058  0.088  0.046
abmil        0.386  0.253  0.055  0.162  0.094  0.051
abmil_wide   0.394  0.238  0.065  0.172  0.083  0.049
transformer  0.395  0.248  0.040  0.162  0.115  0.039
roformer     0.428  0.254  0.054  0.131  0.098  0.036

karolinska_to_radboud: share of slides per grade
grade            0      1      2      3      4      5
true         0.187  0.165  0.131  0.179  0.149  0.189
abmil        0.207  0.151  0.139  0.096  0.296  0.112
abmil_wide   0.211  0.147  0.128  0.115  0.284  0.116
transformer  0.190  0.109  0.154  0.135  0.245  0.166
roformer     0.186  0.114  0.175  0.132  0.240  0.154


## The decision rule

As in notebooks 14 and 15: better in both cross-site directions (interval
above zero), not worse on mixed (interval not below zero).

In [7]:
"""Apply the decision rule to each comparison and print the verdict with its numbers."""

for first, second in pairs:
    name = f"{second} - {first}"
    rows = comparison[(comparison["comparison"] == name) & (comparison["slides"] == "all")].set_index("protocol")
    cross_ok = all(rows.loc[p, "95% low"] > 0 for p in CROSS_SITE)
    mixed_ok = rows.loc["mixed", "95% high"] >= 0
    details = "; ".join(f"{p} {rows.loc[p, 'difference']:+.3f} [{rows.loc[p, '95% low']:+.3f}, "
                        f"{rows.loc[p, '95% high']:+.3f}]" for p in PROTOCOLS)
    print(f"{name:26s} {'helps' if cross_ok and mixed_ok else 'does not help':14s} {details}")

abmil_wide - abmil         does not help  mixed -0.000 [-0.007, +0.006]; radboud_to_karolinska +0.011 [+0.001, +0.021]; karolinska_to_radboud -0.009 [-0.013, -0.004]
roformer - abmil_wide      does not help  mixed +0.010 [+0.003, +0.017]; radboud_to_karolinska +0.077 [+0.064, +0.089]; karolinska_to_radboud -0.005 [-0.014, +0.003]
transformer - abmil_wide   does not help  mixed +0.007 [-0.000, +0.015]; radboud_to_karolinska +0.019 [+0.007, +0.032]; karolinska_to_radboud -0.021 [-0.029, -0.012]


## Result

Run 2026-11-18 on the RTX 4060 PC, packed tissue set `uni2h-L1-224-t144-hsv-v1`.
Wide ABMIL: hidden 2046, 4,205,305 parameters (RoPE model 4,205,319); trained
in 54 minutes.

| Model | Parameters | Mixed | Radboud to Karolinska (5 seeds) | Karolinska to Radboud (5 seeds) |
|---|---|---|---|---|
| ABMIL (baseline) | 1.05 M | 0.864 | 0.632 (0.602 to 0.656) | 0.780 (0.775 to 0.784) |
| wide ABMIL | 4.21 M | 0.864 | 0.642 (0.612 to 0.693) | 0.771 (0.770 to 0.775) |
| transformer + ABMIL | 4.21 M | 0.871 | 0.662 (0.644 to 0.685) | 0.751 (0.732 to 0.773) |
| RoPE transformer + ABMIL | 4.21 M | 0.874 | 0.719 (0.653 to 0.744) | 0.766 (0.753 to 0.777) |

| Comparison | Mixed | Radboud to Karolinska | Karolinska to Radboud |
|---|---|---|---|
| wide ABMIL - ABMIL | -0.000 [-0.007, +0.006] | +0.011 [+0.001, +0.021] | -0.009 [-0.013, -0.004] |
| RoPE - wide ABMIL | +0.010 [+0.003, +0.017] | +0.077 [+0.064, +0.089] | -0.005 [-0.014, +0.003] |
| transformer - wide ABMIL | +0.007 [-0.000, +0.015] | +0.019 [+0.007, +0.032] | -0.021 [-0.029, -0.012] |

- **Capacity alone buys almost nothing.** Four times the parameters in plain
  ABMIL: mixed unchanged, +0.011 one way, -0.009 the other.
- **At equal size, the RoPE model wins on mixed (+0.010) and from Radboud to
  Karolinska (+0.077), and ties from Karolinska to Radboud (-0.005, interval
  across zero).** It misses the strict rule only because that direction is a
  tie rather than a gain. Most of its loss against the small baseline in that
  direction (-0.014) comes with the extra size (-0.009 for the wide ABMIL).
- Self-attention without positions still loses from Karolinska to Radboud
  (-0.021) at equal size; the positions are what make the transformer layer
  work.